# Lab 6

## Task 1

Use beam search to predict the best move in a game of chess by evaluating a subset of the most promising moves. Limit the search depth and beam width for efficiency.
Input: Current board state, beam width, and depth limit.
Output: Best move sequence and its evaluation score.

In [1]:
import chess
import chess.engine
import heapq
from dataclasses import dataclass, field
from typing import List, Tuple, Optional


PIECE_VALUES = {
    chess.PAWN:   100,
    chess.KNIGHT: 320,
    chess.BISHOP: 330,
    chess.ROOK:   500,
    chess.QUEEN:  900,
    chess.KING:     0,
}

PAWN_TABLE = [
     0,  0,  0,  0,  0,  0,  0,  0,
    50, 50, 50, 50, 50, 50, 50, 50,
    10, 10, 20, 30, 30, 20, 10, 10,
     5,  5, 10, 25, 25, 10,  5,  5,
     0,  0,  0, 20, 20,  0,  0,  0,
     5, -5,-10,  0,  0,-10, -5,  5,
     5, 10, 10,-20,-20, 10, 10,  5,
     0,  0,  0,  0,  0,  0,  0,  0,
]

KNIGHT_TABLE = [
    -50,-40,-30,-30,-30,-30,-40,-50,
    -40,-20,  0,  0,  0,  0,-20,-40,
    -30,  0, 10, 15, 15, 10,  0,-30,
    -30,  5, 15, 20, 20, 15,  5,-30,
    -30,  0, 15, 20, 20, 15,  0,-30,
    -30,  5, 10, 15, 15, 10,  5,-30,
    -40,-20,  0,  5,  5,  0,-20,-40,
    -50,-40,-30,-30,-30,-30,-40,-50,
]

BISHOP_TABLE = [
    -20,-10,-10,-10,-10,-10,-10,-20,
    -10,  0,  0,  0,  0,  0,  0,-10,
    -10,  0,  5, 10, 10,  5,  0,-10,
    -10,  5,  5, 10, 10,  5,  5,-10,
    -10,  0, 10, 10, 10, 10,  0,-10,
    -10, 10, 10, 10, 10, 10, 10,-10,
    -10,  5,  0,  0,  0,  0,  5,-10,
    -20,-10,-10,-10,-10,-10,-10,-20,
]

ROOK_TABLE = [
     0,  0,  0,  0,  0,  0,  0,  0,
     5, 10, 10, 10, 10, 10, 10,  5,
    -5,  0,  0,  0,  0,  0,  0, -5,
    -5,  0,  0,  0,  0,  0,  0, -5,
    -5,  0,  0,  0,  0,  0,  0, -5,
    -5,  0,  0,  0,  0,  0,  0, -5,
    -5,  0,  0,  0,  0,  0,  0, -5,
     0,  0,  0,  5,  5,  0,  0,  0,
]

QUEEN_TABLE = [
    -20,-10,-10, -5, -5,-10,-10,-20,
    -10,  0,  0,  0,  0,  0,  0,-10,
    -10,  0,  5,  5,  5,  5,  0,-10,
     -5,  0,  5,  5,  5,  5,  0, -5,
      0,  0,  5,  5,  5,  5,  0, -5,
    -10,  5,  5,  5,  5,  5,  0,-10,
    -10,  0,  5,  0,  0,  0,  0,-10,
    -20,-10,-10, -5, -5,-10,-10,-20,
]

KING_MIDDLE_TABLE = [
    -30,-40,-40,-50,-50,-40,-40,-30,
    -30,-40,-40,-50,-50,-40,-40,-30,
    -30,-40,-40,-50,-50,-40,-40,-30,
    -30,-40,-40,-50,-50,-40,-40,-30,
    -20,-30,-30,-40,-40,-30,-30,-20,
    -10,-20,-20,-20,-20,-20,-20,-10,
     20, 20,  0,  0,  0,  0, 20, 20,
     20, 30, 10,  0,  0, 10, 30, 20,
]

PST_MAP = {
    chess.PAWN:   PAWN_TABLE,
    chess.KNIGHT: KNIGHT_TABLE,
    chess.BISHOP: BISHOP_TABLE,
    chess.ROOK:   ROOK_TABLE,
    chess.QUEEN:  QUEEN_TABLE,
    chess.KING:   KING_MIDDLE_TABLE,
}


# ---------------------------------------------------------------------------
# Static board evaluation (centipawns, positive = good for the side to move)
# ---------------------------------------------------------------------------

def _pst_score(square: int, piece_type: int, color: chess.Color) -> int:
    table = PST_MAP[piece_type]
    idx = square if color == chess.BLACK else chess.square_mirror(square)
    return table[idx]


def evaluate(board: chess.Board) -> int:
    if board.is_checkmate():
        return -100_000
    if board.is_stalemate() or board.is_insufficient_material():
        return 0

    score = 0
    for square, piece in board.piece_map().items():
        value = PIECE_VALUES[piece.piece_type] + _pst_score(square, piece.piece_type, piece.color)
        if piece.color == chess.WHITE:
            score += value
        else:
            score -= value

    return score if board.turn == chess.WHITE else -score


def order_moves(board: chess.Board) -> List[chess.Move]:
    def move_priority(move: chess.Move) -> int:
        priority = 0
        if board.is_capture(move):
            victim = board.piece_at(move.to_square)
            attacker = board.piece_at(move.from_square)
            if victim and attacker:
                priority += 10 * PIECE_VALUES[victim.piece_type] - PIECE_VALUES[attacker.piece_type]
            else:
                priority += 500
        if move.promotion:
            priority += PIECE_VALUES.get(move.promotion, 0)
        board.push(move)
        if board.is_check():
            priority += 50
        board.pop()
        return priority

    moves = list(board.legal_moves)
    moves.sort(key=move_priority, reverse=True)
    return moves


# ---------------------------------------------------------------------------
# Beam node
# ---------------------------------------------------------------------------

@dataclass(order=True)
class BeamNode:
    neg_score: int
    moves: List[chess.Move] = field(compare=False)

    @property
    def score(self) -> int:
        return -self.neg_score


def beam_search(
    board: chess.Board,
    beam_width: int = 5,
    depth_limit: int = 3,
) -> Tuple[List[str], int]:
    if not list(board.legal_moves):
        return [], evaluate(board)

    root_color = board.turn
    initial_node = BeamNode(neg_score=0, moves=[])

    beam: List[BeamNode] = [initial_node]

    best_node = BeamNode(neg_score=0, moves=[])

    for depth in range(depth_limit):
        candidates: List[BeamNode] = []

        for node in beam:
            current_board = board.copy()
            for mv in node.moves:
                current_board.push(mv)

            if current_board.is_game_over():
                leaf_score = evaluate(current_board)
                if len(node.moves) % 2 == 1:
                    leaf_score = -leaf_score
                candidates.append(BeamNode(neg_score=-leaf_score, moves=node.moves))
                continue

            ordered = order_moves(current_board)

            for move in ordered:
                current_board.push(move)
                raw_score = evaluate(current_board)

                plies_played = len(node.moves) + 1
                root_score = raw_score if plies_played % 2 == 0 else -raw_score

                new_moves = node.moves + [move]
                candidates.append(BeamNode(neg_score=-root_score, moves=new_moves))
                current_board.pop()

        if not candidates:
            break

        beam = heapq.nsmallest(beam_width, candidates)

        if beam and beam[0].score > best_node.score:
            best_node = beam[0]

    if not best_node.moves:
        first_move = next(iter(board.legal_moves))
        best_node = BeamNode(neg_score=-evaluate(board), moves=[first_move])

    uci_sequence = [m.uci() for m in best_node.moves]
    return uci_sequence, best_node.score


def print_result(board: chess.Board, moves: List[str], score: int, beam_width: int, depth: int) -> None:
    print("=" * 58)
    print("  Chess Beam Search Result")
    print("=" * 58)
    print(f"  Beam width   : {beam_width}")
    print(f"  Depth limit  : {depth} plies")
    print(f"  Side to move : {'White' if board.turn == chess.WHITE else 'Black'}")
    print("-" * 58)
    print(f"  Best move sequence  : {' → '.join(moves) if moves else '(none)'}")


    tmp = board.copy()
    san_moves = []
    for uci in moves:
        mv = chess.Move.from_uci(uci)
        san_moves.append(tmp.san(mv))
        tmp.push(mv)
    print(f"  In SAN notation     : {' '.join(san_moves)}")
    print(f"  Evaluation score    : {score:+d} cp  ({'White' if score >= 0 else 'Black'} is better)")
    print("=" * 58)


def main() -> None:
    board1 = chess.Board()
    beam_width, depth = 5, 4
    print("\n[Example 1] Starting position")
    print(board1)
    moves1, score1 = beam_search(board1, beam_width=beam_width, depth_limit=depth)
    print_result(board1, moves1, score1, beam_width, depth)

    board2 = chess.Board("r1bqkbnr/pppp1ppp/2n5/4p2Q/2B1P3/8/PPPP1PPP/RNB1K1NR b KQkq - 3 3")
    beam_width2, depth2 = 4, 3
    print("\n[Example 2] Tactical position (Scholar's Mate threat)")
    print(board2)
    moves2, score2 = beam_search(board2, beam_width=beam_width2, depth_limit=depth2)
    print_result(board2, moves2, score2, beam_width2, depth2)

    board3 = chess.Board("4k3/8/8/8/8/8/4K3/4Q3 w - - 0 1")
    beam_width3, depth3 = 3, 5
    print("\n[Example 3] Endgame – King & Queen vs lone King")
    print(board3)
    moves3, score3 = beam_search(board3, beam_width=beam_width3, depth_limit=depth3)
    print_result(board3, moves3, score3, beam_width3, depth3)


if __name__ == "__main__":
    main()



[Example 1] Starting position
r n b q k b n r
p p p p p p p p
. . . . . . . .
. . . . . . . .
. . . . . . . .
. . . . . . . .
P P P P P P P P
R N B Q K B N R
  Chess Beam Search Result
  Beam width   : 5
  Depth limit  : 4 plies
  Side to move : White
----------------------------------------------------------
  Best move sequence  : g1f3 → f7f6 → b1c3 → c7c6
  In SAN notation     : Nf3 f6 Nc3 c6
  Evaluation score    : +140 cp  (White is better)

[Example 2] Tactical position (Scholar's Mate threat)
r . b q k b n r
p p p p . p p p
. . n . . . . .
. . . . p . . Q
. . B . P . . .
. . . . . . . .
P P P P . P P P
R N B . K . N R
  Chess Beam Search Result
  Beam width   : 4
  Depth limit  : 3 plies
  Side to move : Black
----------------------------------------------------------
  Best move sequence  : g8f6 → c4a6 → f6h5
  In SAN notation     : Nf6 Ba6 Nxh5
  Evaluation score    : +955 cp  (White is better)

[Example 3] Endgame – King & Queen vs lone King
. . . . k . . .
. . . . . . . .
.

## Task 2

Design a hill climbing algorithm to find the shortest delivery route among a set of
locations. Make incremental changes to the route and accept them only if they reduce
the total distance.
Input: List of coordinates for delivery points.
Output: Optimized route and the total distance covered.

In [2]:
import math
import random
from itertools import permutations
from typing import List, Tuple


Coord = Tuple[float, float]
Route = List[int]


def euclidean(a: Coord, b: Coord) -> float:
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def total_distance(route: Route, coords: List[Coord]) -> float:
    dist = sum(euclidean(coords[route[i]], coords[route[i + 1]]) for i in range(len(route) - 1))
    dist += euclidean(coords[route[-1]], coords[route[0]])
    return dist


def two_opt_swap(route: Route, i: int, j: int) -> Route:
    return route[:i] + route[i:j + 1][::-1] + route[j + 1:]


def nearest_neighbor_route(coords: List[Coord]) -> Route:
    n = len(coords)
    unvisited = set(range(1, n))
    route = [0]
    while unvisited:
        last = route[-1]
        nearest = min(unvisited, key=lambda idx: euclidean(coords[last], coords[idx]))
        route.append(nearest)
        unvisited.remove(nearest)
    return route


def hill_climbing(coords: List[Coord], restarts: int = 10) -> Tuple[Route, float]:
    n = len(coords)
    best_route = nearest_neighbor_route(coords)
    best_dist = total_distance(best_route, coords)

    for _ in range(restarts):
        current_route = list(range(n))
        random.shuffle(current_route)
        current_dist = total_distance(current_route, coords)

        improved = True
        while improved:
            improved = False
            for i in range(1, n - 1):
                for j in range(i + 1, n):
                    candidate = two_opt_swap(current_route, i, j)
                    candidate_dist = total_distance(candidate, coords)
                    if candidate_dist < current_dist:
                        current_route = candidate
                        current_dist = candidate_dist
                        improved = True

        if current_dist < best_dist:
            best_dist = current_dist
            best_route = current_route

    return best_route, best_dist


def print_result(coords: List[Coord], route: Route, dist: float) -> None:
    print("=" * 52)
    print("  Hill Climbing – Shortest Delivery Route")
    print("=" * 52)
    print(f"  Locations    : {len(coords)}")
    print(f"  Restarts     : 10")
    print("-" * 52)
    route_str = " -> ".join(str(i) for i in route) + f" -> {route[0]}"
    print(f"  Optimized route : {route_str}")
    print(f"  Total distance  : {dist:.4f} units")
    print("=" * 52)


def main() -> None:
    random.seed(42)

    coords_small = [
        (0.0, 0.0),
        (2.0, 4.0),
        (5.0, 2.0),
        (7.0, 5.0),
        (4.0, 8.0),
        (1.0, 6.0),
    ]

    print("\n[Example 1] 6-point delivery grid")
    route1, dist1 = hill_climbing(coords_small, restarts=10)
    print_result(coords_small, route1, dist1)

    coords_medium = [(random.uniform(0, 100), random.uniform(0, 100)) for _ in range(15)]
    print("\n[Example 2] 15 random delivery points")
    route2, dist2 = hill_climbing(coords_medium, restarts=10)
    print_result(coords_medium, route2, dist2)

    coords_large = [(random.uniform(0, 200), random.uniform(0, 200)) for _ in range(30)]
    print("\n[Example 3] 30 random delivery points")
    route3, dist3 = hill_climbing(coords_large, restarts=15)
    print_result(coords_large, route3, dist3)


if __name__ == "__main__":
    main()



[Example 1] 6-point delivery grid
  Hill Climbing – Shortest Delivery Route
  Locations    : 6
  Restarts     : 10
----------------------------------------------------
  Optimized route : 3 -> 4 -> 5 -> 1 -> 0 -> 2 -> 3
  Total distance  : 23.5471 units

[Example 2] 15 random delivery points
  Hill Climbing – Shortest Delivery Route
  Locations    : 15
  Restarts     : 10
----------------------------------------------------
  Optimized route : 7 -> 6 -> 5 -> 9 -> 4 -> 8 -> 3 -> 14 -> 2 -> 11 -> 0 -> 10 -> 12 -> 1 -> 13 -> 7
  Total distance  : 341.4769 units

[Example 3] 30 random delivery points
  Hill Climbing – Shortest Delivery Route
  Locations    : 30
  Restarts     : 10
----------------------------------------------------
  Optimized route : 0 -> 19 -> 14 -> 18 -> 11 -> 8 -> 5 -> 22 -> 3 -> 17 -> 7 -> 9 -> 1 -> 26 -> 15 -> 16 -> 23 -> 6 -> 24 -> 27 -> 2 -> 29 -> 13 -> 20 -> 12 -> 21 -> 28 -> 4 -> 10 -> 25 -> 0
  Total distance  : 916.6770 units


## Task 3

Implement a genetic algorithm to solve the TSP (Traveling Salesman Problem) for a
given set of 10 cities.

In [3]:
import math
import random
from typing import List, Tuple

Coord = Tuple[float, float]
Chromosome = List[int]

CITIES: List[Coord] = [
    (0.0,  0.0),
    (3.0,  4.0),
    (6.0,  1.0),
    (9.0,  5.0),
    (7.0,  9.0),
    (4.0,  7.0),
    (1.0,  5.0),
    (5.0,  3.0),
    (8.0,  2.0),
    (2.0,  8.0),
]

POP_SIZE       = 200
GENERATIONS    = 500
MUTATION_RATE  = 0.02
ELITE_SIZE     = 20
TOURNAMENT_K   = 5


def euclidean(a: Coord, b: Coord) -> float:
    return math.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)


def route_distance(chrom: Chromosome) -> float:
    dist = sum(euclidean(CITIES[chrom[i]], CITIES[chrom[i + 1]]) for i in range(len(chrom) - 1))
    dist += euclidean(CITIES[chrom[-1]], CITIES[chrom[0]])
    return dist


def fitness(chrom: Chromosome) -> float:
    return 1.0 / route_distance(chrom)


def random_chromosome(n: int) -> Chromosome:
    chrom = list(range(n))
    random.shuffle(chrom)
    return chrom


def init_population(n: int) -> List[Chromosome]:
    return [random_chromosome(n) for _ in range(POP_SIZE)]


def tournament_select(population: List[Chromosome]) -> Chromosome:
    contestants = random.sample(population, TOURNAMENT_K)
    return max(contestants, key=fitness)


def ordered_crossover(parent1: Chromosome, parent2: Chromosome) -> Chromosome:
    n = len(parent1)
    a, b = sorted(random.sample(range(n), 2))
    segment = parent1[a:b + 1]
    child = [-1] * n
    child[a:b + 1] = segment
    fill = [gene for gene in parent2 if gene not in segment]
    idx = 0
    for i in range(n):
        if child[i] == -1:
            child[i] = fill[idx]
            idx += 1
    return child


def swap_mutate(chrom: Chromosome) -> Chromosome:
    chrom = chrom[:]
    if random.random() < MUTATION_RATE:
        i, j = random.sample(range(len(chrom)), 2)
        chrom[i], chrom[j] = chrom[j], chrom[i]
    return chrom


def next_generation(population: List[Chromosome]) -> List[Chromosome]:
    sorted_pop = sorted(population, key=fitness, reverse=True)
    elites = sorted_pop[:ELITE_SIZE]
    offspring = []
    while len(offspring) < POP_SIZE - ELITE_SIZE:
        p1 = tournament_select(population)
        p2 = tournament_select(population)
        child = ordered_crossover(p1, p2)
        child = swap_mutate(child)
        offspring.append(child)
    return elites + offspring


def genetic_algorithm() -> Tuple[Chromosome, float]:
    n = len(CITIES)
    population = init_population(n)
    best_chrom = min(population, key=route_distance)
    best_dist = route_distance(best_chrom)

    for gen in range(1, GENERATIONS + 1):
        population = next_generation(population)
        current_best = min(population, key=route_distance)
        current_dist = route_distance(current_best)
        if current_dist < best_dist:
            best_dist = current_dist
            best_chrom = current_best[:]
        if gen % 100 == 0:
            print(f"  Generation {gen:>4} | Best distance: {best_dist:.4f}")

    return best_chrom, best_dist


def print_result(route: Chromosome, dist: float) -> None:
    print("=" * 52)
    print("  Genetic Algorithm – TSP (10 Cities)")
    print("=" * 52)
    print(f"  Population size : {POP_SIZE}")
    print(f"  Generations     : {GENERATIONS}")
    print(f"  Mutation rate   : {MUTATION_RATE}")
    print(f"  Elite size      : {ELITE_SIZE}")
    print("-" * 52)
    route_str = " -> ".join(str(c) for c in route) + f" -> {route[0]}"
    print(f"  Best route      : {route_str}")
    print(f"  Total distance  : {dist:.4f} units")
    print("=" * 52)
    print("\n  City coordinates:")
    for i, (x, y) in enumerate(CITIES):
        print(f"    City {i}: ({x}, {y})")
    print("=" * 52)


def main() -> None:
    random.seed(42)
    print("\nRunning Genetic Algorithm for TSP...\n")
    best_route, best_dist = genetic_algorithm()
    print()
    print_result(best_route, best_dist)


if __name__ == "__main__":
    main()



Running Genetic Algorithm for TSP...

  Generation  100 | Best distance: 33.4455
  Generation  200 | Best distance: 33.4455
  Generation  300 | Best distance: 33.4455
  Generation  400 | Best distance: 33.4455
  Generation  500 | Best distance: 33.4455

  Genetic Algorithm – TSP (10 Cities)
  Population size : 200
  Generations     : 500
  Mutation rate   : 0.02
  Elite size      : 20
----------------------------------------------------
  Best route      : 8 -> 2 -> 7 -> 1 -> 0 -> 6 -> 9 -> 5 -> 4 -> 3 -> 8
  Total distance  : 33.4455 units

  City coordinates:
    City 0: (0.0, 0.0)
    City 1: (3.0, 4.0)
    City 2: (6.0, 1.0)
    City 3: (9.0, 5.0)
    City 4: (7.0, 9.0)
    City 5: (4.0, 7.0)
    City 6: (1.0, 5.0)
    City 7: (5.0, 3.0)
    City 8: (8.0, 2.0)
    City 9: (2.0, 8.0)


## Task 4

Design a Beam Search algorithm to allocate job tasks to available processors in a
distributed system. The algorithm should consider factors such as execution time,
processor load, and priority.
Input: A list of job tasks with execution times and priorities, and a set of available
processors.
Output: Optimized task allocation that minimizes the maximum load on any processor.

In [4]:
import heapq
from dataclasses import dataclass, field
from typing import List, Tuple, Dict


@dataclass
class Task:
    task_id: int
    exec_time: float
    priority: int


@dataclass(order=True)
class State:
    score: float
    loads: Tuple[float, ...] = field(compare=False)
    assignment: List[int] = field(compare=False)
    step: int = field(compare=False)


def heuristic(loads: Tuple[float, ...], remaining_tasks: List[Task]) -> float:
    max_load = max(loads)
    avg_load = sum(loads) / len(loads)
    imbalance = max_load - avg_load
    future_pressure = sum(t.exec_time / (t.priority + 1) for t in remaining_tasks)
    return max_load + 0.4 * imbalance + 0.1 * future_pressure


def beam_search(
    tasks: List[Task],
    num_processors: int,
    beam_width: int = 10,
) -> Tuple[List[int], float]:
    sorted_tasks = sorted(tasks, key=lambda t: (-t.priority, -t.exec_time))

    initial_loads = tuple(0.0 for _ in range(num_processors))
    initial_state = State(
        score=0.0,
        loads=initial_loads,
        assignment=[],
        step=0,
    )

    beam: List[State] = [initial_state]
    best_state: State = initial_state

    for step in range(len(sorted_tasks)):
        task = sorted_tasks[step]
        candidates: List[State] = []

        for state in beam:
            seen_loads = set()
            for proc in range(num_processors):
                new_load = state.loads[proc] + task.exec_time
                key = round(new_load, 6)
                if key in seen_loads:
                    continue
                seen_loads.add(key)

                new_loads = list(state.loads)
                new_loads[proc] = new_load
                new_loads_t = tuple(new_loads)

                remaining = sorted_tasks[step + 1:]
                score = heuristic(new_loads_t, remaining)

                new_assignment = state.assignment + [proc]
                candidates.append(State(
                    score=score,
                    loads=new_loads_t,
                    assignment=new_assignment,
                    step=step + 1,
                ))

        beam = heapq.nsmallest(beam_width, candidates)

        if beam:
            local_best = min(beam, key=lambda s: max(s.loads))
            if not best_state.assignment or max(local_best.loads) < max(best_state.loads):
                best_state = local_best

    if not best_state.assignment:
        return [], 0.0

    return best_state.assignment, max(best_state.loads)


def build_allocation(
    tasks: List[Task],
    assignment: List[int],
    num_processors: int,
) -> Dict[int, List[Task]]:
    sorted_tasks = sorted(tasks, key=lambda t: (-t.priority, -t.exec_time))
    allocation: Dict[int, List[Task]] = {p: [] for p in range(num_processors)}
    for task, proc in zip(sorted_tasks, assignment):
        allocation[proc].append(task)
    return allocation


def print_result(
    tasks: List[Task],
    assignment: List[int],
    makespan: float,
    num_processors: int,
    beam_width: int,
) -> None:
    allocation = build_allocation(tasks, assignment, num_processors)
    sorted_tasks = sorted(tasks, key=lambda t: (-t.priority, -t.exec_time))

    print("=" * 60)
    print("  Beam Search – Distributed Task Allocation")
    print("=" * 60)
    print(f"  Tasks          : {len(tasks)}")
    print(f"  Processors     : {num_processors}")
    print(f"  Beam width     : {beam_width}")
    print(f"  Makespan       : {makespan:.4f} time units")
    print("-" * 60)

    for p in range(num_processors):
        task_list = allocation[p]
        load = sum(t.exec_time for t in task_list)
        ids = ", ".join(f"T{t.task_id}(p={t.priority},t={t.exec_time})" for t in task_list)
        ids = ids if ids else "(idle)"
        print(f"  Processor {p}  | Load: {load:6.2f} | Tasks: {ids}")

    print("-" * 60)
    print("  Task → Processor mapping (sorted by priority):")
    for task, proc in zip(sorted_tasks, assignment):
        print(f"    T{task.task_id:>2} (priority={task.priority}, exec={task.exec_time:.2f}) → Processor {proc}")
    print("=" * 60)


def main() -> None:
    tasks_ex1 = [
        Task(0, exec_time=5.0,  priority=3),
        Task(1, exec_time=3.0,  priority=5),
        Task(2, exec_time=8.0,  priority=1),
        Task(3, exec_time=2.0,  priority=4),
        Task(4, exec_time=6.0,  priority=2),
        Task(5, exec_time=4.0,  priority=5),
        Task(6, exec_time=7.0,  priority=1),
    ]
    num_proc1, bw1 = 3, 8
    print("\n[Example 1] 7 tasks → 3 processors")
    assignment1, makespan1 = beam_search(tasks_ex1, num_proc1, beam_width=bw1)
    print_result(tasks_ex1, assignment1, makespan1, num_proc1, bw1)

    tasks_ex2 = [
        Task(0,  exec_time=10.0, priority=5),
        Task(1,  exec_time=4.0,  priority=3),
        Task(2,  exec_time=7.0,  priority=4),
        Task(3,  exec_time=2.0,  priority=2),
        Task(4,  exec_time=9.0,  priority=5),
        Task(5,  exec_time=3.0,  priority=1),
        Task(6,  exec_time=6.0,  priority=3),
        Task(7,  exec_time=5.0,  priority=4),
        Task(8,  exec_time=8.0,  priority=2),
        Task(9,  exec_time=1.0,  priority=5),
        Task(10, exec_time=11.0, priority=1),
        Task(11, exec_time=3.5,  priority=3),
    ]
    num_proc2, bw2 = 4, 10
    print("\n[Example 2] 12 tasks → 4 processors")
    assignment2, makespan2 = beam_search(tasks_ex2, num_proc2, beam_width=bw2)
    print_result(tasks_ex2, assignment2, makespan2, num_proc2, bw2)


if __name__ == "__main__":
    main()



[Example 1] 7 tasks → 3 processors
  Beam Search – Distributed Task Allocation
  Tasks          : 7
  Processors     : 3
  Beam width     : 8
  Makespan       : 4.0000 time units
------------------------------------------------------------
  Processor 0  | Load:   4.00 | Tasks: T5(p=5,t=4.0)
  Processor 1  | Load:   0.00 | Tasks: (idle)
  Processor 2  | Load:   0.00 | Tasks: (idle)
------------------------------------------------------------
  Task → Processor mapping (sorted by priority):
    T 5 (priority=5, exec=4.00) → Processor 0

[Example 2] 12 tasks → 4 processors
  Beam Search – Distributed Task Allocation
  Tasks          : 12
  Processors     : 4
  Beam width     : 10
  Makespan       : 10.0000 time units
------------------------------------------------------------
  Processor 0  | Load:  10.00 | Tasks: T0(p=5,t=10.0)
  Processor 1  | Load:   0.00 | Tasks: (idle)
  Processor 2  | Load:   0.00 | Tasks: (idle)
  Processor 3  | Load:   0.00 | Tasks: (idle)
---------------------